# Conversation Management: the built-in default for context engineering

**What is conversation management?** It is the built-in strategy an agent uses to curate its conversation history so the context window stays small and relevant. Before reaching for advanced patterns (memory pointers, RAG, multi-agent), Strands Agents already ships this for you.

This notebook compares the four built-in strategies on the **same conversation** and the **same recall question**, measuring native token usage each time:

| Strategy | What it does |
|---|---|
| **None** (baseline) | Keep the full history, nothing is dropped |
| **Sliding Window** | Keep only the most recent messages |
| **Summarization** | Summarize older messages, keep few (or no) recent ones |
| **Combination** | Summary of older messages **+** recent ones kept verbatim |

> This demo uses Strands Agents. Sliding window, summarization, and combination are general agent concepts and carry over to other agent frameworks.

## Setup

Install dependencies and set your `OPENAI_API_KEY` (in a `.env` file or as an environment variable). Get a key at [platform.openai.com/api-keys](https://platform.openai.com/api-keys).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os
from dotenv import load_dotenv
from strands import Agent
# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel
from strands.agent.conversation_manager import (
    NullConversationManager,
    SlidingWindowConversationManager,
    SummarizingConversationManager,
)

# Suppress noisy OpenTelemetry warnings
os.environ["OTEL_SDK_DISABLED"] = "true"

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not set. Get your API key from https://platform.openai.com/api-keys "
        "then add OPENAI_API_KEY=your-key to a .env file, or export it in your shell."
    )

MODEL = OpenAIModel(model_id="gpt-4o-mini")
print("Ready.")

## The probe

The user states a key fact in **turn 1** — the deploy budget is **$500/month** — then 11 filler turns bury it. At the end we ask *"what is my budget?"* and check whether each strategy still remembers, and how many tokens the query costs.

The conversation is seeded directly into `agent.messages` (no LLM calls), so the setup is deterministic and free. The only real model calls are the recall question and, for the summarizing strategies, the one-time summary generation.

In [ ]:
RECALL_QUESTION = (
    "What is my monthly deploy budget? "
    "Reply with only the dollar amount, or say 'unknown' if it is not in our conversation."
)

# The key fact stated in turn 1. Sliding Window will trim it; the summarizing
# strategies should preserve it.
KEY_FACT = "500"

CONVERSATION = [
    ("Our monthly deploy budget is $500. Keep that in mind for everything.",
     "Understood — I'll keep your $500/month deploy budget in mind."),
    ("We ship a Python service on AWS.", "Got it, a Python service on AWS."),
    ("The team has four engineers.", "Noted, four engineers."),
    ("We use GitHub Actions for CI.", "Understood, GitHub Actions for CI."),
    ("Production runs in us-east-1.", "Noted, production in us-east-1."),
    ("Staging runs in us-west-2.", "Got it, staging in us-west-2."),
    ("We deploy on Fridays.", "Understood, Friday deploys."),
    ("Alerts go to the #ops Slack channel.", "Noted, alerts to #ops."),
    ("We keep 30 days of logs.", "Got it, 30-day log retention."),
    ("Our database is Postgres.", "Understood, Postgres."),
    ("We cache with Redis.", "Noted, Redis for caching."),
    ("Load tests run every Monday.", "Got it, Monday load tests."),
]


def seed_history(agent):
    """Append the fixed conversation to an agent's history without calling the LLM."""
    for user_msg, assistant_msg in CONVERSATION:
        agent.messages.append({"role": "user", "content": [{"text": user_msg}]})
        agent.messages.append({"role": "assistant", "content": [{"text": assistant_msg}]})


def total_tokens(response):
    """Total tokens for a query, from Strands' native metrics (provider-agnostic).

    accumulated_usage[\"totalTokens\"] = input (system prompt + tools + managed
    history + question) + output (answer). It can be None if a provider does not
    report usage, so we guard for that.
    """
    if response.metrics and response.metrics.accumulated_usage:
        return response.metrics.accumulated_usage["totalTokens"]
    return 0

print(f"{len(CONVERSATION)} turns seeded. Budget fact is in turn 1.")

## One function, four strategies

For each strategy we build the full history, let the manager curate it, then ask the recall question on a fresh agent carrying only the managed history — so the native token count reflects exactly that one query's context size.

Summarizing managers compact **reactively** (on real context overflow). Here we call `reduce_context()` explicitly so the compaction is visible without needing a full 128K-token overflow.

In [ ]:
def run_strategy(name, manager, summarize):
    print("\n" + "=" * 70)
    print(f"STRATEGY: {name}")
    print("=" * 70)

    # 1. Build the full history, then let the manager curate it.
    staging = Agent(model=MODEL, conversation_manager=manager)
    seed_history(staging)
    before = len(staging.messages)

    if summarize:
        staging.conversation_manager.reduce_context(staging)
    else:
        staging.conversation_manager.apply_management(staging)

    managed = list(staging.messages)
    after = len(managed)
    print(f"History: {before} messages → {after} after management")

    # Is the budget fact still anywhere in the kept context?
    kept_text = " ".join(
        block.get("text", "")
        for msg in managed
        for block in msg.get("content", [])
        if isinstance(block, dict)
    )
    print(f"Budget fact still in context: {'yes' if KEY_FACT in kept_text else 'no'}")

    # 2. Ask the recall question on a fresh agent carrying only the managed history.
    #    callback_handler=None keeps the streamed answer out of the output.
    recall_agent = Agent(model=MODEL, callback_handler=None)
    recall_agent.messages = managed
    response = recall_agent(RECALL_QUESTION)

    answer = str(response).strip()
    tokens = total_tokens(response)
    remembered = KEY_FACT in answer
    print(f"Answer: {answer}")
    print(f"Remembered budget: {'✅ yes' if remembered else '❌ no'}")
    print(f"📊 Query tokens (native): {tokens:,}")

    return {"name": name, "tokens": tokens, "remembered": remembered, "kept": after}

### 1. None (baseline) — keep the full history

Nothing is dropped. The agent remembers everything, but every query pays for the entire conversation.

In [ ]:
r_none = run_strategy(
    "None (baseline)",
    NullConversationManager(),
    summarize=False,
)

### 2. Sliding Window — keep only the most recent messages

`window_size=6` keeps the last 6 messages. Cheapest context — but the budget fact from turn 1 is trimmed away, so recall fails.

In [ ]:
r_sliding = run_strategy(
    "Sliding Window (window_size=6)",
    SlidingWindowConversationManager(window_size=6),
    summarize=False,
)

### 3. Summarization — summarize old, keep none recent

`summary_ratio=0.8, preserve_recent_messages=0` compresses the whole history into a summary. Compact, and the budget fact survives inside the summary.

In [ ]:
r_summary = run_strategy(
    "Summarization (ratio=0.8, preserve=0)",
    SummarizingConversationManager(summary_ratio=0.8, preserve_recent_messages=0),
    summarize=True,
)

### 4. Combination — summarize old + keep recent

`summary_ratio=0.5, preserve_recent_messages=4` keeps a summary of older turns **and** the most recent turns verbatim. This is the recommended default: compact context that still remembers the early fact.

In [ ]:
r_combination = run_strategy(
    "Combination (ratio=0.5, preserve=4)",
    SummarizingConversationManager(summary_ratio=0.5, preserve_recent_messages=4),
    summarize=True,
)

## Comparison

Same conversation, same question. Note token counts vary slightly between runs because the summary is generated live by the model.

In [ ]:
results = [r_none, r_sliding, r_summary, r_combination]

print(f"{'Strategy':<40} {'Kept':>5} {'Tokens':>9} {'Recall':>8}")
print("-" * 64)
for r in results:
    recall = "✅" if r["remembered"] else "❌"
    print(f"{r['name']:<40} {r['kept']:>5} {r['tokens']:>9,} {recall:>7}")

print("\nTakeaways:")
print("  • None    — remembers everything, but every query pays for the full history.")
print("  • Sliding — cheapest context, but trims the early budget fact → wrong answer.")
print("  • Summarize / Combine — compact AND keep the fact. Combination is the")
print("    recommended default: a summary of old turns plus recent turns kept verbatim.")

## When to use which

- **Sliding Window** — short-lived tasks where only recent context matters and old turns are safe to forget. Fast and cheap, no extra model call.
- **Summarization** — long conversations where old details still matter but do not need to be verbatim.
- **Combination** — the recommended general-purpose default: recent turns exact, older turns summarized. Set it with `SummarizingConversationManager(summary_ratio=..., preserve_recent_messages=...)`.

Conversation management is the *starting point*. When a single tool returns a large indivisible blob (logs, documents) that would overflow the window in one shot, move on to the **Memory Pointer Pattern** in [`01-context-overflow-demo`](../01-context-overflow-demo/).